# Exploration du référentiel des installations

Ce notebook vérifie la structure du référentiel M02 et sa relation avec le fichier horaire des urgences. La source officielle est la [Liste des installations du MSSS](https://www.donneesquebec.ca/recherche/dataset/fichiers-cartographiques-m02-des-installations-et-etablissements/resource/2aa06e66-c1d0-4e2f-bf3c-c2e413c3f84d).

In [1]:
from hashlib import sha256
from pathlib import Path

import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INSTALLATIONS_FILE = PROJECT_ROOT / "data" / "exploration" / "installations.csv"
EMERGENCY_FILE = PROJECT_ROOT / "data" / "exploration" / "urgences_horaires.csv"

print(f"Référentiel trouvé : {INSTALLATIONS_FILE.exists()}")
print(f"Fichier des urgences trouvé : {EMERGENCY_FILE.exists()}")

Référentiel trouvé : True
Fichier des urgences trouvé : True


## Chargement du référentiel

Les colonnes sont chargées comme du texte afin de préserver les codes administratifs et leurs éventuels zéros initiaux.

In [3]:
installations_raw = pd.read_csv(
    INSTALLATIONS_FILE,
    encoding="utf-8-sig",
    encoding_errors="strict",
    dtype="string",
)

installations = installations_raw.copy()
installations.columns = installations.columns.str.strip()
installations = installations.apply(lambda column: column.str.strip())
installations = installations.replace("", pd.NA)

print(f"Nombre de lignes : {len(installations):,}")
print(f"Nombre de colonnes : {len(installations.columns)}")
print(f"SHA-256 : {sha256(INSTALLATIONS_FILE.read_bytes()).hexdigest()}")

Nombre de lignes : 1,589
Nombre de colonnes : 44
SHA-256 : 916ee240bd0705ab711ac2b966a568a70bb958214c41308cc7409305dd28854f


In [4]:
key_profile = pd.Series(
    {
        "lignes": len(installations),
        "codes_distincts": installations["INSTAL_COD"].nunique(),
        "codes_manquants": installations["INSTAL_COD"].isna().sum(),
        "lignes_avec_code_duplique": installations.duplicated(
            subset=["INSTAL_COD"],
            keep=False,
        ).sum(),
    },
    name="valeur",
)

key_profile.to_frame()

,valeur
lignes,1589
codes_distincts,1589
codes_manquants,0
lignes_avec_code_duplique,0


In [5]:
installations[
    [
        "INSTAL_COD",
        "INSTAL_NOM",
        "ETAB_CODE",
        "ETAB_NOM",
        "RSS_CODE",
        "RSS_NOM",
        "MUN_NOM",
        "LONGITUDE",
        "LATITUDE",
        "DATE_MAJ",
    ]
].head()

,INSTAL_COD,INSTAL_NOM,ETAB_CODE,ETAB_NOM,RSS_CODE,RSS_NOM,MUN_NOM,LONGITUDE,LATITUDE,DATE_MAJ
0,12387692,CHSLD POLONAIS MARIE-CURIE-SKLODOWSKA,11045200,Santé Québec Est-de-l'Île-de-Montréal – Univer...,06,Montréal,Montréal,-73.57430267333984,45.58100128173828,2026-09-14
1,12475976,HÔPITAL DU SACRÉ-COEUR-DE-MONTRÉAL,11045192,Santé Québec Nord-de-l'Île-de-Montréal – Unive...,06,Montréal,Montréal,-73.71399688720703,45.53260040283203,2026-09-14
2,12934659,HÔPITAL MAISONNEUVE-ROSEMONT,11045200,Santé Québec Est-de-l'Île-de-Montréal – Univer...,06,Montréal,Montréal,-73.55850219726562,45.57419967651367,2026-09-14
3,15103666,INSTITUT UNIVERSITAIRE DE GÉRIATRIE DE MONTRÉAL,11045184,Santé Québec Centre-Sud-de-l'Île-de-Montréal –...,06,Montréal,Montréal,-73.6238021850586,45.49100112915039,2026-09-14
4,51216281,SERVICES PROFESSIONNELS - MRC MITIS,11045119,Santé Québec Bas-Saint-Laurent,01,Bas-Saint-Laurent,Mont-Joli,-68.1969985961914,48.59120178222656,2026-09-14


## Relation avec les urgences

La jointure est testée entre `No_permis_installation` dans les urgences et `INSTAL_COD` dans le référentiel. Elle est effectuée en texte pour préserver exactement les identifiants publiés.

In [6]:
emergency = pd.read_csv(
    EMERGENCY_FILE,
    encoding="cp1252",
    encoding_errors="strict",
    dtype="string",
)
emergency.columns = emergency.columns.str.strip()
emergency = emergency.apply(lambda column: column.str.strip())
emergency = emergency.replace("", pd.NA)
emergency_installations = emergency.loc[
    emergency["No_permis_installation"].notna()
].copy()

In [7]:
facility_reference = installations[
    [
        "INSTAL_COD",
        "INSTAL_NOM",
        "ETAB_CODE",
        "ETAB_NOM",
        "RSS_CODE",
        "RSS_NOM",
        "MUN_NOM",
        "LONGITUDE",
        "LATITUDE",
        "DATE_MAJ",
    ]
]

facility_match = emergency_installations.merge(
    facility_reference,
    how="left",
    left_on="No_permis_installation",
    right_on="INSTAL_COD",
    validate="many_to_one",
    indicator=True,
)

facility_match["_merge"].value_counts().to_frame(name="lignes")

,lignes
_merge,
both,120
left_only,0
right_only,0


In [8]:
matched = facility_match.loc[facility_match["_merge"].eq("both")].copy()

relationship_checks = pd.Series(
    {
        "urgences_sans_correspondance": facility_match["_merge"].ne("both").sum(),
        "noms_installation_differents": (
            matched["Nom_installation"].str.casefold()
            != matched["INSTAL_NOM"].str.casefold()
        ).sum(),
        "codes_region_differents": (
            matched["RSS"].astype("Int64")
            != matched["RSS_CODE"].astype("Int64")
        ).sum(),
    },
    name="nombre_de_cas",
)

relationship_checks.to_frame()

,nombre_de_cas
urgences_sans_correspondance,0
noms_installation_differents,0
codes_region_differents,0


## Conclusion

- `INSTAL_COD` est renseigné et unique dans le référentiel observé.
- Les 120 installations du fichier des urgences trouvent une correspondance dans le référentiel.
- Les noms d'installation et les codes de région concordent pour les 120 correspondances.
- La relation `No_permis_installation` vers `INSTAL_COD` est donc retenue pour la première version du modèle.
- Cette relation devra rester couverte par un test de qualité à chaque chargement, car les référentiels administratifs peuvent évoluer.